## Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

## Loading dataset

In [ ]:
df = pd.read_csv('Nepali_Treking_EnhancedV2.csv')

### Data Understanding

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(df['Country'].value_counts())
print(df['Regional code'].value_counts())

In [ ]:
#missing values
df.isnull().sum()

In [ ]:
df.isnull().sum()/df.shape[0]*100

In [ ]:
#duplicates
df.duplicated().sum()

In [ ]:
for i in df.select_dtypes(include="object").columns:
  print(df[i].unique())
  print("*****"*10)

In [ ]:
#garbage values
for i in df.select_dtypes(include="object").columns:
  print(df[i].value_counts())
  print("*****"*10)

In [ ]:
df.describe(include='object')

In [ ]:
#histogram to understand distribution
import warnings 
warnings.filterwarnings("ignore")
for i in df.select_dtypes(include="number").columns:
    sns.histplot(data=df, x=i)
    plt.show()

In [ ]:
#histogram to understand distribution
import warnings 
warnings.filterwarnings("ignore")
for i in df.select_dtypes(include="number").columns:
    sns.boxplot(data=df, x=i)
    plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

#correlation matrix
corr = numeric_df.corr()
corr

In [ ]:
# heatmap
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Trekking Features')
plt.show()


## Cleaning dataset

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'Health Incidents','Equipment Used','GraduateOrNot','Employment Type','FrequentFlyer','Regional code','Country'])
df.isnull().sum()

In [ ]:
# Remove leading/trailing whitespaces from column names
df.columns = df.columns.str.strip()

In [ ]:
#Renaming columns
df = df.rename({"Trip Grade":"Grade"}, axis=1)
df = df.rename({"Trekking Group Size":"GroupSize"}, axis=1)
df = df.rename({"Guide/No Guide":"Guide"}, axis=1)
df = df.rename({"Review/Satisfaction":"Review"}, axis=1)

##### Handling missing values

In [ ]:
df.fillna({'GroupSize':df['GroupSize'].median()}, inplace=True)

In [ ]:
df['GroupSize'].unique()

In [ ]:
df['GroupSize']=df['GroupSize'].astype(int)

In [ ]:
df.fillna({'Review':df['Review'].mean()}, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
# Contingency table to show the relationship between Trip Grade and Guide/No Guide
contingency_table_guide = pd.crosstab(df['GroupSize'], df['Guide'])
print(contingency_table_guide)

In [ ]:
from scipy.stats import chi2_contingency

# Performing the Chi-Square test on the contingency table
chi2, p, dof, expected = chi2_contingency(contingency_table_guide)

print(f'Chi-Square Statistic: {chi2}')
print(f'P-Value: {p}')

In [ ]:
# Function to fill NaN in 'Guide/No Guide' based on the most frequent value (mode) for each group size
def fill_guide_based_on_group_size(row):
    if pd.isnull(row['Guide']):
        # Get the most frequent value (mode) for the corresponding group size
        mode_guide = df[df['GroupSize'] == row['GroupSize']]['Guide'].mode()[0]
        return mode_guide
    else:
        return row['Guide']

# Apply the function to fill NaN values
df['Guide'] = df.apply(fill_guide_based_on_group_size, axis=1)

In [ ]:
df['Guide'].isnull().sum()

In [ ]:
print(df['Guide'].unique())

In [ ]:
df['Fitness Level'].unique()

In [ ]:
contingency_table_fitness = pd.crosstab(df['Grade'], df['Fitness Level'])
print(contingency_table_fitness)

In [ ]:
# Performing the Chi-Square test on the contingency table
chi2, p, dof, expected = chi2_contingency(contingency_table_fitness)

print(f'Chi-Square Statistic: {chi2}')
print(f'P-Value: {p}')

In [ ]:
df.columns

In [ ]:
from scipy.stats import f_oneway

# Group the data by Fitness Level
groups = [group['Age'].values for name, group in df.groupby('Fitness Level')]

# Perform one-way ANOVA
f_stat, p_value = f_oneway(*groups)

print(f'ANOVA F-Statistic: {f_stat}')
print(f'P-Value: {p_value}')

In [ ]:
mode_fitness = df['Fitness Level'].mode()[0]
df['Fitness Level'].fillna(mode_fitness, inplace=True)

In [ ]:
df["Fitness Level"].unique()

In [ ]:
df["Grade"].unique()

In [ ]:
def clean_grade(x):
    if 'Easy To Moderate' in x or 'Easy-Moderate' in x or 'Light+Moderate' in x:
        return 'Easy to Moderate'
    if 'Light' in x or 'Easy' in x:
        return 'Easy'
    if 'Moderate' in x:
        return 'Moderate'
    return 'Hard'
df["Grade"] = df['Grade'].apply(clean_grade)

In [ ]:
df['Grade'].unique()

In [ ]:
df['Cost'].unique()

In [ ]:
# Cleaning 'Cost' column and converting to float
df['Cost'] = df['Cost'].str.replace('\n', '').str.replace('USD', '').str.replace('$', '').str.replace(',', '').str.strip()
df['Cost']=df['Cost'].astype(float)
df['Cost'].head()

In [ ]:
df['Max Altitude'].unique()

In [ ]:
df['Max Altitude'] = df['Max Altitude'].str.replace('m', '').str.replace(',', '').str.strip()
df['Max Altitude']=df['Max Altitude'].astype(int)
df['Max Altitude'].unique()

In [ ]:
df['Year'].unique()

In [ ]:
mode_year = df['Year'].mode()[0]
mode_year

In [ ]:
Q1 = df['Year'].quantile(0.25)
Q3 = df['Year'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Replacing outliers with the mode
df.loc[(df['Year'] < lower_bound) | (df['Year'] > upper_bound), 'Year'] = mode_year

df['Year'].unique()

In [ ]:
df['Time'].unique()

In [ ]:
#cleaning time column and changing data type
df['Time'] = df['Time'].apply(lambda x: re.sub(r'\b[Dd]ays?\b', '', x)).astype(int)

In [ ]:
#rechecking datatypes
df.dtypes

In [ ]:
def clean_best_time(time):
    time = time.replace('March - Nov', 'March - May & Sept - Nov')
    return time
df['Best Travel Time'] = df['Best Travel Time'].apply(clean_best_time)
df['Best Travel Time'].unique()

In [ ]:
# Spliting 'Best Travel Time' into two columns and assigning seasons
df[['Travel Time 1', 'Travel Time 2']] = df['Best Travel Time'].str.split('&', expand=True)

In [ ]:
df['Travel Time 1'].unique()

In [ ]:
df['Travel Time 2'].unique()

In [ ]:
# function to clean the 'Travel Time' columns
def clean_travel_time(time):
    if pd.isnull(time):
        return None
    time = time.strip()  # Removing leading and trailing spaces
    time = time.replace('Setpt', 'Sept')  # For fixing typo
    time = time.replace(' - ', '-')
    time = time.replace('.', '')
    time = time.replace('- ', '-')# Standardizing hyphen usage
    time = re.sub(r'(\s+)', ' ', time)  # Removing extra spaces
    return time

# Applying the cleaning function to 'Travel Time 1' and 'Travel Time 2'
df['Travel Time 1'] = df['Travel Time 1'].apply(clean_travel_time)
df['Travel Time 2'] = df['Travel Time 2'].apply(clean_travel_time)

print(df['Travel Time 1'].unique())
print(df['Travel Time 2'].unique())

In [ ]:
print(df[df['Travel Time 2'].isnull()])

In [ ]:
# Mapping months to seasons
def map_season(time):
    if time in ['March-May', 'April-May', 'Jan-May']:
        return 'Spring'
    elif time == 'June-Aug':
        return 'Summer'
    elif time in ['Sept-Dec', 'Sept-Nov']:
        return 'Autumn'
    elif time in ['March-Nov']:
        return 
    elif time == 'Dec-Feb':
        return 'Winter'
    else:
        return None

df['Travel Season 1'] = df['Travel Time 1'].apply(map_season)
df['Travel Season 2'] = df['Travel Time 2'].apply(map_season)

# Drop the original travel time columns as they're no longer needed
df = df.drop(columns=['Travel Time 1', 'Travel Time 2'])

print(df[['Travel Season 1', 'Travel Season 2']].head())

In [ ]:
# Function to clean trek names
def clean_trek_name(name):
    if pd.isnull(name):  # Handle NaN values
        return name
    name = name.strip()  # Remove leading/trailing whitespace
    name = name.replace('\xa0', ' ')  # Replace non-breaking spaces with regular spaces
    name = ' '.join(name.split())  # Remove extra spaces between words
    return name

# Apply the cleaning function to the 'Trek' column
df['Trek'] = df['Trek'].apply(clean_trek_name)

# Check unique values after cleaning
unique_treks_cleaned = df['Trek'].unique()
print(unique_treks_cleaned)

In [ ]:
# Function to clean accommodation values
def clean_accommodation(value):
    value = value.strip().lower()  # Normalize case and remove leading/trailing spaces
    if 'guesthouse' in value or 'guest houses' in value or 'guesthouses' in value:
        return 'Hotel/Guesthouse'
    elif 'luxury' in value:
        return 'Hotel/Luxury Lodge'
    elif 'teahouse' in value or 'teahouses' in value:
        return 'Hotel/Teahouse'
    elif 'lodges' in value or 'lodge' in value:
        return 'Hotel/Lodge'
    else:
        return value.capitalize()  # Capitalize the first letter of remaining values for consistency

# Apply the cleaning function to the 'Accommodation' column
df['Accomodation'] = df['Accomodation'].apply(clean_accommodation)

# Verify the cleaned values with a value count
print(df['Accomodation'].value_counts())


In [ ]:
df.columns

### Visualizing features

In [ ]:
pivot_data=df.pivot_table(index='Grade',columns='Fitness Level',values='Max Altitude')

In [ ]:
#heatmap of the data
sns.heatmap(pivot_data,annot=True)
plt.title('Data Heatmap')

In [ ]:
# 'Fitness Level' to a numeric format 
df['Fitness Level'] = df['Fitness Level'].astype('category').cat.codes

# the correlation
correlation = df[['Cost', 'Fitness Level']].corr()

print(correlation)

plt.figure(figsize=(10, 6))
sns.boxplot(x='Fitness Level', y='Cost', data=df)
plt.title('Relationship between Cost and Fitness Level')
plt.xlabel('Fitness Level')
plt.ylabel('Grade')
plt.show()

In [ ]:
# Distribution of costs
plt.figure(figsize=(10,6))
sns.histplot(df['Cost'], kde=True)
plt.title('Distribution of Trek Costs')
plt.show()

In [ ]:
# Cost vs Group Size with Satisfaction as color
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='GroupSize', y='Cost', hue='Review', palette='coolwarm')
plt.title('Trekking Group Size vs Cost (Colored by Satisfaction)')
plt.show()

In [ ]:
# Count the occurrences of each 'Trek'
trek_counts = df['Trek'].value_counts()

# Set the figure size for better visibility
plt.figure(figsize=(12, 6))

# Create a bar plot to visualize the counts of each 'Trek'
sns.barplot(x=trek_counts.index, y=trek_counts.values)

# Set the title and labels
plt.title('Counts of Each Trek')
plt.xlabel('Count')
plt.ylabel('Trek')

# Rotate x labels for better readability if there are many treks
plt.xticks(rotation=90)

# Show the plot
plt.show()

In [ ]:
# Get the top 10 treks by count
top_treks = trek_counts.head(10)

# Plotting the top 10 treks
plt.figure(figsize=(12, 6))
sns.barplot(x=top_treks.values, y=top_treks.index)
plt.title('Top 10 Treks by Count')
plt.xlabel('Count')
plt.ylabel('Trek Name')
plt.show()


In [ ]:
# Grouping by Age and Most visited Trek to find the average satisfaction
age_trek_satisfaction = df.groupby(['Age', (df['Trek'].head(10))])['Review'].mean().reset_index()

# Visualization
plt.figure(figsize=(12, 6))
sns.barplot(data=age_trek_satisfaction, x='Age', y='Review', hue='Trek')
plt.title('Average Trek Satisfaction by Age Group')
plt.ylabel('Average Satisfaction')
plt.xlabel('Age')
plt.xticks(rotation=45)
plt.legend(title='Trek')
plt.show()

In [ ]:
# Group by Travel Season and calculate average satisfaction
seasonal_satisfaction = df.groupby(['Accomodation'])['Review'].mean().reset_index()

# Visualization
plt.figure(figsize=(10, 6))
sns.barplot(data=seasonal_satisfaction, x='Accomodation', y='Review')
plt.title('Average Trek Satisfaction by Accomodation')
plt.ylabel('Average Satisfaction')
plt.xlabel('Accomodation')
plt.show()

In [ ]:
# Create the stacked bar plot
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='GroupSize', hue='Guide', palette='coolwarm')

# Set titles and labels
plt.title('Group Size vs Guide Usage')
plt.xlabel('Group Size')
plt.ylabel('Count of Trekkers')
plt.show()


In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 7))
df.boxplot('Cost', 'Accomodation', ax=ax)
plt.suptitle('Cost (US$) v Accomodation')
plt.title('')
plt.ylabel('Cost')
plt.xticks(rotation=90)
plt.show()

### Data preparation and splitting

In [ ]:
df.columns

In [ ]:
from sklearn.preprocessing import LabelEncoder
le_grade = LabelEncoder()
df["Grade"] = le_grade.fit_transform(df["Grade"])
df["Grade"].unique()

In [ ]:
le_accomodation = LabelEncoder()
df['Accomodation'] = le_accomodation.fit_transform(df['Accomodation'])
df["Accomodation"].unique()

In [ ]:
le_sex = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex'])

In [ ]:
df['Sex'].unique()

In [ ]:
le_guide = LabelEncoder()
df['Guide'] = le_guide.fit_transform(df['Guide'])
df["Guide"].unique()

In [ ]:
X= df[["Time", "Grade", "Accomodation","Sex", "Age","GroupSize","Guide"]]
y= df["Cost"]
print(X.shape)
print(y.shape)

In [ ]:
# Set up the plot
plt.figure(figsize=(15, 10))

# Create scatter plots for each feature against y
features = ["Time", "Grade", "Accomodation", "Sex", "Age", "GroupSize", "Guide"]
for i, feature in enumerate(features):
    plt.subplot(3, 3, i + 1)  # Create a grid of plots
    plt.scatter(df[feature], y, alpha=0.5)
    plt.title(f'Scatter plot of {feature} vs Cost')
    plt.xlabel(feature)
    plt.ylabel('Cost')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_test

### Model Building

##### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
linear_reg = LinearRegression()
linear_reg.fit(X_train, y_train)

In [ ]:
y_lr_train_pred = linear_reg.predict(X_train)
y_lr_test_pred= linear_reg.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lr_train_mse = mean_squared_error(y_train, y_lr_train_pred)
lr_train_r2 = r2_score(y_train, y_lr_train_pred)
lr_test_mse = mean_squared_error(y_test, y_lr_test_pred)
lr_test_r2 = r2_score(y_test, y_lr_test_pred)

In [ ]:
print('LR MSE (Train): ', lr_train_mse)
print('LR R2 (Train): ', lr_train_r2)
print('LR MSE (Test): ', lr_test_mse)
print('LR R2 (Test): ', lr_test_r2)

In [ ]:
error = np.sqrt(mean_squared_error(y_train, y_lr_train_pred))
error

In [ ]:
lr_results = pd.DataFrame(['Linear regression', lr_train_mse, lr_train_r2, lr_test_mse, lr_test_r2]).transpose()
lr_results.columns = ['Method', 'Training MSE', 'Training R2', 'Test MSE', 'Test R2']
lr_results

In [ ]:
from sklearn.tree import DecisionTreeRegressor
dec_tree_reg = DecisionTreeRegressor(max_depth = 3, random_state=0)
dec_tree_reg.fit(X_train, y_train)

In [ ]:
y_dt_train_pred = dec_tree_reg.predict(X_train)
y_dt_test_pred = dec_tree_reg.predict(X_test)

In [ ]:
dt_train_mse = mean_squared_error(y_train, y_dt_train_pred)
dt_train_r2 = r2_score(y_train, y_dt_train_pred)

dt_test_mse = mean_squared_error(y_test, y_dt_test_pred)
dt_test_r2 = r2_score(y_test, y_dt_test_pred)
dt_error = np.sqrt(mean_squared_error(y_train, y_dt_train_pred))

In [ ]:
print('DT MSE (Train): ', dt_train_mse)
print('DT R2 (Train): ', dt_train_r2)
print('DT MSE (Test): ', dt_test_mse)
print('DT R2 (Test): ', dt_test_r2)
print("${:,.02f}".format(dt_error))

In [ ]:
dt_results = pd.DataFrame(['Decision Tree', dt_train_mse, dt_train_r2, dt_test_mse, dt_test_r2]).transpose()
dt_results.columns = ['Method', 'Training MSE', 'Training R2', 'Test MSE', 'Test R2']
dt_results

In [ ]:
from sklearn.ensemble import RandomForestRegressor
random_forest_reg = RandomForestRegressor(max_depth=2,random_state=0)
random_forest_reg.fit(X_train, y_train)

In [ ]:
y_rf_train_pred = random_forest_reg.predict(X_train)
y_rf_test_pred = random_forest_reg.predict(X_test)

In [ ]:
rf_train_mse = mean_squared_error(y_train, y_rf_train_pred)
rf_train_r2 = r2_score(y_train, y_rf_train_pred)

rf_test_mse = mean_squared_error(y_test, y_rf_test_pred)
rf_test_r2 = r2_score(y_test, y_rf_test_pred)
rf_error = np.sqrt(mean_squared_error(y_train, y_rf_train_pred))

In [ ]:
print('LR MSE (Train): ', rf_train_mse)
print('LR R2 (Train): ', rf_train_r2)
print('LR MSE (Test): ', rf_test_mse)
print('LR R2 (Test): ', rf_test_r2)
print("${:,.02f}".format(rf_error))

In [ ]:
rf_results = pd.DataFrame(['Random forest', rf_train_mse, rf_train_r2, rf_test_mse, rf_test_r2]).transpose()
rf_results.columns = ['Method', 'Training MSE', 'Training R2', 'Test MSE', 'Test R2']
rf_results

In [ ]:
df_models = pd.concat([lr_results, dt_results, rf_results], axis=0)
df_models

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(x=y_train, y=y_lr_train_pred, c="#7CAE00" ,alpha=0.3)

z = np.polyfit(y_train, y_lr_train_pred, 1)
p = np.poly1d(z)

plt.plot(y_train, p(y_train), '#F8766D')
plt.ylabel('Predict LogS')
plt.xlabel('Experimental LogS')

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(x=y_train, y=y_dt_train_pred, c="#7CAE00" ,alpha=0.3)

z = np.polyfit(y_train, y_dt_train_pred, 1)
p = np.poly1d(z)

plt.plot(y_train, p(y_train), '#F8766D')
plt.ylabel('Predict LogS')
plt.xlabel('Experimental LogS')

In [ ]:
X= df[["Time", "Grade", "Accomodation","Sex", "Age","GroupSize","Guide"]]
y= df["Cost"]
from sklearn.model_selection import GridSearchCV

max_depth = [None, 2,4,6,8,10,12]
parameters = {"max_depth": max_depth}

regressor = DecisionTreeRegressor(random_state=0)
gs = GridSearchCV(regressor, parameters, scoring='neg_mean_squared_error')
gs.fit(X, y.values)

In [ ]:
X

In [ ]:
regressor = DecisionTreeRegressor(random_state=0)

regressor.fit(X, y.values)
y_pred = regressor.predict(X)
error = np.sqrt(mean_squared_error(y, y_pred))
print("${:,.02f}".format(error))

In [ ]:
X

In [ ]:
df["Grade"].unique()

In [ ]:
# Input data
X = np.array([[3, "Easy", "Hotel/Guesthouse", "Male", 30, 5, "No Guide"]])
X

In [ ]:
X[:, 1] = le_grade.transform(X[:,1])
X[:, 2] = le_accomodation.transform(X[:,2])
X[:, 3] = le_sex.transform(X[:,3])
X[:, 6] = le_guide.transform(X[:,6])
X = X.astype(float)
X

In [ ]:
y_pred = regressor.predict(X)
y_pred

In [ ]:
df.to_csv('cleaned_data.csv', index=False)

#### Recommendation System

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler 

In [ ]:
# Scaling the dataset's numerical features
df_scaled = df.copy()

# Only scale numerical columns (including 'Cost' now)
numerical_features = ['Time', 'Age', 'GroupSize', 'Cost']
df_scaled[numerical_features] = scaler.fit_transform(df_scaled[numerical_features])

In [ ]:
# Features used for similarity: ['Duration', 'Grade', 'Accommodation', 'Sex', 'Age', 'Group_Size', 'Guide', 'Cost']
df_features = df_scaled[['Time', 'Grade', 'Accomodation', 'Sex', 'Age', 'GroupSize', 'Guide', 'Cost']].values

In [ ]:
# Split data into training and test sets
x_train, x_test = train_test_split(df_features, test_size=0.2, random_state=42)

In [ ]:
# Training the KNN model on the training set
knn_model = NearestNeighbors(n_neighbors=5, metric='euclidean')
knn_model.fit(x_train)

In [ ]:
# Evaluate on the test set
test_user_input = x_test[0].reshape(1, -1) 
# Find the 5 nearest neighbors in the test set
distances, indices = knn_model.kneighbors(test_user_input)

In [ ]:
# Get the recommended treks based on the test set
recommended_treks = df.iloc[indices[0]]['Trek'].values
print("Recommended Treks:", recommended_treks)

In [ ]:
# Check Cosine Similarity with the recommended treks
recommended_features = x_train[indices[0]]
cos_sim = cosine_similarity(test_user_input, recommended_features)
print("Cosine Similarity with recommended treks:", cos_sim)

In [ ]:
# User input for KNN (example)
user_input = np.array([[5, "Moderate", "Hotel/Guesthouse", "Male", 30, 5, "Guide", 1500]])

In [ ]:
# label encoding
user_input[:, 1] = le_grade.transform(user_input[:, 1].astype(str))
user_input[:, 2] = le_accomodation.transform(user_input[:, 2].astype(str))
user_input[:, 3] = le_sex.transform(user_input[:, 3].astype(str))
user_input[:, 6] = le_guide.transform(user_input[:, 6].astype(str))
user_input = user_input.astype(float)

# Find the 5 nearest treks and their distances
distances, indices = knn_model.kneighbors(user_input)

# Output the distances
print("Distances from input to recommended treks:", distances)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
# feature vectors of the recommended treks
recommended_features = df_features[indices[0]]
#cosine similarity between user input and recommended treks
cos_sim = cosine_similarity(user_input, recommended_features)
print("Cosine Similarity with recommended treks:", cos_sim)

In [ ]:
X_user_scaled = np.array([[12, 1, 1, 1, 30, 5, 1,590]])
distances,indices = knn_model.kneighbors(X_user_scaled)

In [ ]:
import pickle

In [ ]:
data = {"model": regressor, "knn_model": knn_model, "scaler":scaler,"le_grade": le_grade, "le_accomodation": le_accomodation, "le_sex": le_sex, "le_guide":le_guide}
with open('saved_steps.pkl', 'wb') as file:
    pickle.dump(data, file)

In [ ]:
with open('saved_steps.pkl', 'rb') as file:
    data = pickle.load(file)

regressor_loaded = data["model"]
le_grade = data["le_grade"]
le_accomodation = data["le_accomodation"]
le_sex = data["le_sex"]
le_guide = data["le_guide"]

In [ ]:
y_pred = regressor_loaded.predict(X)
y_pred

In [ ]:
df['Trek'].unique()

In [ ]:
df.loc[(df['Time'] == 25)]